In [ ]:
import pandas as pd
import requests
import notebook
import sqlalchemy
import psycopg2
from retry_requests import retry
from sqlalchemy import create_engine
from sqlalchemy import text 

In [14]:
import os
API_KEY = os.environ.get("API_KEY", "REDACTED")
BASE_URL = "https://api.collegebasketballdata.com/#/games/GetGames"
HEADERS = {'Authorization': f'Bearer {API_KEY}', 'accept': 'application/json'}

In [16]:
def fetch_endpoint(endpoint, API_KEY, params=None):
   
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "accept": "application/json"
    }

    url = f"{BASE_URL}{endpoint}"
    response = requests.get(url, headers=headers, params=params)

    print("Request URL:", response.url)
    print("Status Code:", response.status_code)

    response.raise_for_status()
    return response.json()


def fetch_basketball_data(API_KEY, startDate='2018-08-01', endDate='2025-12-31'):
    

    games = fetch_endpoint(
        "/games",
        API_KEY=API_KEY,
        params={
            "startDate": startDate,
            "endDate": endDate
        }
    )

    game_rows = []
    player_rows = []

    for game in games:
        game_id = game.get("id")
        game_date = game.get("date")
        season = game.get("season")

        home_team = game.get("homeTeam", {})
        away_team = game.get("awayTeam", {})

        home_score = home_team.get("score", 0)
        away_score = away_team.get("score", 0)

        home_win = 1 if home_score > away_score else 0
        away_win = 1 if away_score > home_score else 0

        # Team-game rows
        game_rows.append({
            "game_id": game_id,
            "date": game_date,
            "season": season,
            "team_id": home_team.get("id"),
            "team": home_team.get("name"),
            "opponent_id": away_team.get("id"),
            "opponent": away_team.get("name"),
            "location": "home",
            "points_for": home_score,
            "points_against": away_score,
            "win": home_win
        })

        game_rows.append({
            "game_id": game_id,
            "date": game_date,
            "season": season,
            "team_id": away_team.get("id"),
            "team": away_team.get("name"),
            "opponent_id": home_team.get("id"),
            "opponent": home_team.get("name"),
            "location": "away",
            "points_for": away_score,
            "points_against": home_score,
            "win": away_win
        })

        # Player stats (if box score exists)
        try:
            box = fetch_endpoint(f"/games/{game_id}/boxscore", API_KEY=API_KEY)

            for team_box in box.get("teams", []):
                team_id = team_box.get("teamId")
                team_name = team_box.get("teamName")

                for player in team_box.get("players", []):
                    stats = player.get("stats", {})

                    player_rows.append({
                        "game_id": game_id,
                        "date": game_date,
                        "season": season,
                        "team_id": team_id,
                        "team": team_name,
                        "player_id": player.get("id"),
                        "player": player.get("name"),
                        "minutes": stats.get("minutes"),
                        "points": stats.get("points"),
                        "rebounds": stats.get("rebounds"),
                        "assists": stats.get("assists"),
                        "steals": stats.get("steals"),
                        "blocks": stats.get("blocks"),
                        "turnovers": stats.get("turnovers"),
                        "fouls": stats.get("fouls"),
                        "fgm": stats.get("fieldGoalsMade"),
                        "fga": stats.get("fieldGoalsAttempted"),
                        "tpm": stats.get("threePointsMade"),
                        "tpa": stats.get("threePointsAttempted"),
                        "ftm": stats.get("freeThrowsMade"),
                        "fta": stats.get("freeThrowsAttempted")
                    })

        except requests.HTTPError:
            continue

    games_df = pd.DataFrame(game_rows)
    player_stats_df = pd.DataFrame(player_rows)

    # ---- Team yearly record ----
    team_record_df = (
        games_df.groupby(["season", "team_id", "team"], as_index=False)
        .agg(
            games_played=("game_id", "count"),
            wins=("win", "sum")
        )
    )

    team_record_df["losses"] = team_record_df["games_played"] - team_record_df["wins"]
    team_record_df["win_pct"] = (
        team_record_df["wins"] / team_record_df["games_played"]
    ).round(3)

    team_record_df["record"] = (
        team_record_df["wins"].astype(str) + "-" + team_record_df["losses"].astype(str)
    )

    # ---- Home / Away record ----
    location_record_df = (
        games_df.groupby(["season", "team_id", "team", "location"], as_index=False)
        .agg(
            games_played=("game_id", "count"),
            wins=("win", "sum")
        )
    )

    location_record_df["losses"] = (
        location_record_df["games_played"] - location_record_df["wins"]
    )

    location_record_df["record"] = (
        location_record_df["wins"].astype(str) + "-" + location_record_df["losses"].astype(str)
    )

    return games_df, player_stats_df, team_record_df, location_record_df


  